In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jessicali9530/caltech256")

print("Path to dataset files:", path)

100%|██████████| 2.12G/2.12G [01:44<00:00, 21.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jessicali9530/caltech256/versions/2


In [ ]:
import os

path = '/root/.cache/kagglehub/datasets/jessicali9530/caltech256/versions/2/256_objectcategories/256_ObjectCategories/'
files = os.listdir(path)
print(files)


['087.goldfish', '176.saddle', '094.guitar-pick', '054.diamond-ring', '010.beer-mug', '066.ewer-101', '051.cowboy-hat', '125.knife', '052.crab-101', '043.coin', '196.spaghetti', '127.laptop-101', '030.canoe', '251.airplanes-101', '146.mountain-bike', '122.kayak', '186.skunk', '213.teddy-bear', '191.sneaker', '233.tuning-fork', '142.microwave', '199.spoon', '223.top-hat', '217.tennis-court', '203.stirrups', '239.washing-machine', '174.rotary-phone', '178.school-bus', '022.buddha-101', '161.photocopier', '158.penguin', '082.galaxy', '224.touring-bike', '214.teepee', '208.swiss-army-knife', '165.pram', '032.cartman', '187.skyscraper', '226.traffic-light', '073.fireworks', '257.clutter', '016.boom-box', '159.people', '011.billiards', '140.menorah-101', '168.raccoon', '118.iris', '230.trilobite-101', '192.snowmobile', '100.hawksbill-101', '001.ak47', '190.snake', '234.tweezer', '162.picnic-table', '147.mushroom', '215.telephone-box', '058.doorknob', '170.rainbow', '107.hot-air-balloon', '02

In [ ]:
import os
print(os.path.exists(path))  # Check if path exists
print(os.access(path, os.R_OK))  # Check if we have read permissions

True
True


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision import models
from torch.utils.data import DataLoader
import pandas as pd
import os
from PIL import Image

# Define constants
NUM_CLASSES = 256
BATCH_SIZE = 32
LEARNING_RATE = 0.0002
NUM_EPOCHS = 2019
TRAIN_DIR = "/root/.cache/kagglehub/datasets/jessicali9530/caltech256/versions/2/256_objectcategories/256_ObjectCategories/"  # 257 directories including parent
OUTPUT_CSV = "sample_submission.csv"

def get_data_loaders(train_dir, batch_size):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
    # Subtract 1 from all labels to shift the range from [1, 256] to [0, 255]
    train_dataset.targets = [target - 1 for target in train_dataset.targets]

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    return train_loader, train_dataset.classes

# Load dataset
train_loader, class_names = get_data_loaders(TRAIN_DIR, BATCH_SIZE)


In [ ]:
for images, labels in train_loader:
    print(f"Min label: {labels.min()}, Max label: {labels.max()}")  # Must be in range [0, 255]
    break


Min label: 15, Max label: 256


In [ ]:
for images, labels in train_loader:
    print(labels.dtype)  # Must be torch.int64 (long)
    break

torch.int64


In [ ]:

# Load Pretrained VGG19 model
model = models.vgg19(pretrained=True)
for param in model.parameters():
    param.requires_grad = False  # Freeze all layers

for param in model.features[-2:].parameters():
    param.requires_grad = True

lr_features = 1e-5  # Lower LR for fine-tuned layers
lr_classifier = 1e-3  # Higher LR for newly added classifier

params = [
    {"params": model.features[-2:].parameters(), "lr": lr_features},  # Fine-tuning layers
    {"params": model.classifier.parameters(), "lr": lr_classifier}  # Classifier
]

# Modify the classifier for 256 classes
model.classifier[6] = nn.Sequential(
    nn.Linear(4096, 512),  # Input: 4096, Output: 512
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 257)  # Input: 512, Output: 256 (for 256 classes)
)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(params)

def train_model(model, train_loader, criterion, optimizer, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            labels = labels
            images, labels = images.to("cuda"), labels.to("cuda")
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")

# Move model to GPU
model = model.to("cuda")
train_model(model, train_loader, criterion, optimizer, NUM_EPOCHS)
# Save the model after training
torch.save(model.state_dict(), "vgg19_custom_model.pth")



/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:04<00:00, 130MB/s]


Epoch 1/10, Loss: 2.3730137564296383
Epoch 2/10, Loss: 1.2010838502128545
Epoch 3/10, Loss: 1.0301616782616423
Epoch 4/10, Loss: 0.9233952887753334
Epoch 5/10, Loss: 0.8622586077221259
Epoch 6/10, Loss: 0.8033580142564783
Epoch 7/10, Loss: 0.7626316757286853
Epoch 8/10, Loss: 0.7286424458244876
Epoch 9/10, Loss: 0.701790753981661
Epoch 10/10, Loss: 0.6739275298366477


In [ ]:
import zipfile
import os

# Path to the ZIP file
zip_path = "/content/to-classify-images-given-in-caltech-256-dataset.zip"  # Replace with your actual ZIP file name

# Get the directory where the ZIP file is located
extract_dir = os.path.dirname(zip_path)

# Extract the ZIP file into the same directory
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Files extracted to {extract_dir}")


Files extracted to /content


In [ ]:
TEST_DIR = "/content/test"  # Separate test images folder

def predict_and_generate_csv(model, test_dir, output_csv):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    image_paths = []
    predictions = []
    for img_name in os.listdir(test_dir):
        img_path = os.path.join(test_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        image = transform(image).unsqueeze(0).to("cuda")
        with torch.no_grad():
            output = model(image)
            predicted_label = class_names[torch.argmax(output).item()]
        image_paths.append(img_name)
        predictions.append(predicted_label)
    df = pd.DataFrame({"img_path": image_paths, "label": predictions})
    df.to_csv(output_csv, index=False)
    print("CSV file generated successfully!")

predict_and_generate_csv(model, TEST_DIR, OUTPUT_CSV)

CSV file generated successfully!


In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv("/content/sample_submission.csv")

# Modify the 'label' column, splitting by '.' and taking the second part, then capitalizing
df['label'] = df['label'].apply(lambda x: x.split('.')[1] if '.' in x else x).str.title()

# Modify the 'img_path' column
df['img_path'] = df['img_path'].apply(lambda x: f'test/{x}')

# Save the modified DataFrame to a new CSV file
df.to_csv("final.csv", index=False)
